# nb/04c: fetch the picked C2C12 frames (B2 parity, B8 demo set)

Written by `scripts/pick_c2c12_frames.py`. Follows the Drive I/O rule: copy the one archive to local disk, extract only the listed files there, check each file's SHA-256 against the cache, and copy one small zip back to Drive. Then download `staged/c2c12_picks.zip` to the Mac and unzip it into `data/c2c12_picks/`.

In [ ]:
import hashlib, os, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')
STAGE_DIR = '/content/drive/MyDrive/cultureqc/staged'   # same folder nb/04b used; edit if yours differs
FILES = {
 "9fb8e6e8b2cf05396a99fd6c90cebeab32c542585c6eb683e401351076a401ca": "/content/data/c2c12/png/c2c12_090318_exp1_F0003_Data/00589.png",
 "35d466eb11aad0759843dd4f58173d424821b4b68d6619994c10e84f7f95abb6": "/content/data/c2c12/png/c2c12_090325_exp1_F0003_Data/00373.png",
 "d859aad2ba840be1ece5803022c91b81f97d66e191825e766563b5ceb557a179": "/content/data/c2c12/png/c2c12_090325_exp1_F0013_Data/00889.png",
 "26d74d8a47f174a76b9d78c27e510555550be5d7fdce23716413a689441386d2": "/content/data/c2c12/png/c2c12_090318_exp1_F0011_Data/00433.png",
 "fdbbf70aa4216bbc7cf8c827a91df276d15eab0371675306863a95b8b11c475a": "/content/data/c2c12/png/c2c12_090318_exp1_F0013_Data/01045.png",
 "9605c29f71614c064bd8481e48729272fdca0aff4d0e581591e48cd614debca5": "/content/data/c2c12/png/c2c12_090303_exp1_F0003_Data/00913.png",
 "92fad07b5d16a1e0872ac5cd07ef468983ab2ad39fee51829a3ad0e5e7df03bf": "/content/data/c2c12_faults/png/c2c12_090325_exp1_F0013_Data__fault_contam/00637.png",
 "b5f67decd6da40888f688722f2d8ffe99a97417ec9785d24ff1be6afe7fd6d1b": "/content/data/c2c12_faults/png/c2c12_090318_exp1_F0005_Data__fault_contam/00769.png",
 "d8d75a864d57486ac3803342a8be1e928483255911774962b8956b20b692bde5": "/content/data/c2c12_faults/png/c2c12_090318_exp1_F0016_Data__fault_dim/00841.png",
 "00ee04b2cc49a7a0b234de40cfd2382938b474179b8120401ec0f5a508c89631": "/content/data/c2c12_faults/png/c2c12_090318_exp1_F0003_Data__fault_dim/00841.png"
}

local_zip = '/content/c2c12_prepared.zip'
shutil.copy(f'{STAGE_DIR}/c2c12_prepared.zip', local_zip)      # one sequential read from Drive
os.makedirs('/content/picks', exist_ok=True)
with zipfile.ZipFile(local_zip) as z:
    names = {'/' + n.lstrip('/'): n for n in z.namelist()}
    for sha, path in FILES.items():
        data = z.read(names[path])
        got = hashlib.sha256(data).hexdigest()
        assert got == sha, f'{path}: sha256 {got} != cache {sha}'
        open(f'/content/picks/{sha}.png', 'wb').write(data)
os.remove(local_zip)
shutil.make_archive('/content/c2c12_picks', 'zip', '/content/picks')
shutil.copy('/content/c2c12_picks.zip', f'{STAGE_DIR}/c2c12_picks.zip')
print(len(FILES), 'frames verified and copied to', f'{STAGE_DIR}/c2c12_picks.zip')
